# Worker A: Healthcare (All Conditions)

Runs the full Obermeyer healthcare grid in one session.
Healthcare uses analytic (KKT-based) gradients — no finite-difference overhead.

**Grid:** 7 methods × 2 alphas × 5 seeds = 70 runs  
**Est. time:** ~30-45 min on Colab GPU

Checkpoint/resume: re-run any cell to continue — completed runs are skipped.

In [ ]:
import os
import sys

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ----- Path setup -----
DRIVE_ROOT = '/content/drive/MyDrive/DecisionFocusedMTL'
MOSEK_LIC_PATH = f'{DRIVE_ROOT}/data/mosek.lic'

# Set MOSEK license
if os.path.exists(MOSEK_LIC_PATH):
    os.environ['MOSEKLM_LICENSE_FILE'] = MOSEK_LIC_PATH
    print(f'MOSEK license set: {MOSEK_LIC_PATH}')
else:
    print(f'Warning: MOSEK license not found at {MOSEK_LIC_PATH}')

# Add source paths
for p in [DRIVE_ROOT, os.path.join(DRIVE_ROOT, 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

%cd {DRIVE_ROOT}
print(f'Working directory: {os.getcwd()}')

# ----- Results directory -----
HC_RESULTS = os.path.join(DRIVE_ROOT, 'results', 'final', 'healthcare')
os.makedirs(HC_RESULTS, exist_ok=True)

# ----- Imports -----
from experiments.colab_runner import *
import torch
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')
print(f'Results: {HC_RESULTS}')

In [ ]:
# ===== Experiment Configuration =====
# Change these to adjust parameters without touching source files.
# Pass {} for any override dict to use all defaults.

# Task-level overrides (healthcare decision problem)
TASK_OVERRIDES = {
    # "n_sample": 0,          # 0 = use all 48,784 patients
    # "budget_rho": 0.35,     # budget = rho * sum(capped_costs)
    # "test_fraction": 0.5,
    # "val_fraction": 0.0,
    # "decision_mode": "group",
    # "fairness_type": "mad",
}

# Training-level overrides (optimizer, model architecture)
TRAIN_OVERRIDES = {
    # "lr": 0.0005,
    # "lr_decay": 0.0005,
    # "grad_clip_norm": 10000.0,
    # "log_every": 5,
    # --- model architecture (routed into model sub-config) ---
    # "hidden_dim": 64,
    # "n_layers": 2,
    # "activation": "relu",
    # "dropout": 0.0,
    # "batch_norm": False,
}

STEPS = 70

print(f"Steps per lambda: {STEPS}")
if TASK_OVERRIDES:
    print(f"Task overrides: {TASK_OVERRIDES}")
if TRAIN_OVERRIDES:
    print(f"Train overrides: {TRAIN_OVERRIDES}")

In [ ]:
run_healthcare_slice(
    methods=['FPTO', 'SAA', 'WDRO'],
    results_dir=HC_RESULTS,
    steps=STEPS,
    task_overrides=TASK_OVERRIDES,
    train_overrides=TRAIN_OVERRIDES,
)

In [ ]:
run_healthcare_slice(
    methods=['FPTO', 'SAA', 'WDRO'],
    results_dir=HC_RESULTS,
)

In [ ]:
show_progress(HC_RESULTS, 'Healthcare — after baselines')

In [ ]:
run_healthcare_slice(
    methods=['FDFL-Scal'],
    results_dir=HC_RESULTS,
    steps=STEPS,
    task_overrides=TASK_OVERRIDES,
    train_overrides=TRAIN_OVERRIDES,
)

In [ ]:
run_healthcare_slice(
    methods=['FDFL-Scal'],
    results_dir=HC_RESULTS,
)

In [ ]:
show_progress(HC_RESULTS, 'Healthcare — after FDFL-Scal')

In [ ]:
run_healthcare_slice(
    methods=['FDFL-PCGrad', 'FDFL-MGDA', 'FDFL-CAGrad'],
    results_dir=HC_RESULTS,
    steps=STEPS,
    task_overrides=TASK_OVERRIDES,
    train_overrides=TRAIN_OVERRIDES,
)

In [ ]:
run_healthcare_slice(
    methods=['FDFL-PCGrad', 'FDFL-MGDA', 'FDFL-CAGrad'],
    results_dir=HC_RESULTS,
)

In [ ]:
show_progress(HC_RESULTS, 'Healthcare — COMPLETE')

Worker A complete. Run the Results notebook to analyze.